In [8]:
import numpy as np
import pandas as pd


def descrever_dados(
    df,
    coluna=None,
    coluna_grupo=None,
    remover_outliers=False,
    salvar_arquivo=True,
    formato_arquivo="excel",
    nome_arquivo="resumo_estatistico",
):
    """Analisa colunas numéricas de um DataFrame e opcionalmente exporta os

    resultados.

    Parâmetros:
    -----------
    df : pandas.DataFrame
        O DataFrame contendo os dados.
    coluna : str, opcional
        Nome da coluna numérica específica a ser analisada. Se None, analisa
        todas.
    coluna_grupo : str, opcional
        Nome da coluna para agrupamento dos dados (ex: 'Amostra', 'Categoria').
    remover_outliers : bool, default False
        Se True, retorna a tabela descritiva e uma cópia do DataFrame sem
        outliers.
    salvar_arquivo : bool, default True
        Se True, salva a tabela resumo descritiva em um arquivo.
    formato_arquivo : str, default 'excel'
        Formato do arquivo exportado: 'excel' (.xlsx) ou 'csv' (.csv).
    nome_arquivo : str, default 'resumo_estatistico'
        Nome base do arquivo a ser salvo (sem a extensão).

    Retorna:
    --------
    tabela : pandas.DataFrame
        Tabela descritiva formatada.
    df_limpo : pandas.DataFrame (apenas se remover_outliers=True)
        DataFrame filtrado sem os outliers das colunas analisadas.
    """
    if df is None or not isinstance(df, pd.DataFrame):
        raise ValueError(
            "O parâmetro 'df' deve ser um pandas.DataFrame válido."
        )

    # Define quais colunas numéricas serão processadas
    if coluna:
        if coluna not in df.columns:
            raise ValueError(
                f"A coluna '{coluna}' não foi encontrada no DataFrame."
            )
        colunas_analise = [coluna]
    else:
        colunas_analise = df.select_dtypes(include=[np.number]).columns.tolist()
        if coluna_grupo and coluna_grupo in colunas_analise:
            colunas_analise.remove(coluna_grupo)

        if not colunas_analise:
            print(
                "⚠️ Nenhuma coluna numérica foi encontrada no DataFrame para"
                " análise."
            )
            return None

    # Função interna para calcular estatísticas por coluna/grupo
    def calcular_estatisticas(s):
        s_clean = s.dropna()
        n = len(s_clean)
        if n == 0:
            return pd.Series({
                "Total": 0,
                "Média": np.nan,
                "Mediana": np.nan,
                "Desvio Padrão": np.nan,
                "Q1 (25%)": np.nan,
                "Q3 (75%)": np.nan,
                "IQR": np.nan,
                "Outliers (Qtd)": 0,
                "Outliers (Valores)": "[]",
            })

        media = s_clean.mean()
        mediana = s_clean.median()
        std = s_clean.std(ddof=1)

        q1 = s_clean.quantile(0.25)
        q3 = s_clean.quantile(0.75)
        iqr = q3 - q1

        lim_inferior = q1 - 1.5 * iqr
        lim_superior = q3 + 1.5 * iqr

        outliers = s_clean[
            (s_clean < lim_inferior) | (s_clean > lim_superior)
        ].tolist()

        return pd.Series({
            "Total": n,
            "Média": round(media, 4),
            "Mediana": round(mediana, 4),
            "Desvio Padrão": round(std, 4),
            "Q1 (25%)": round(q1, 4),
            "Q3 (75%)": round(q3, 4),
            "IQR": round(iqr, 4),
            "Outliers (Qtd)": len(outliers),
            "Outliers (Valores)": str(outliers)
            if len(outliers) > 0
            else "Nenhum",
        })

    # 1. Agrupamento ou Análise Geral
    if coluna_grupo:
        if coluna_grupo not in df.columns:
            raise ValueError(
                f"A coluna de agrupamento '{coluna_grupo}' não foi encontrada."
            )

        tabelas = []
        for col in colunas_analise:
            tb = (
                df.groupby(coluna_grupo)[col]
                .apply(calcular_estatisticas)
                .unstack()
            )
            tb.insert(0, "Coluna", col)
            tabelas.append(tb)
        tabela = pd.concat(tabelas)
        print(f"📊 --- RESUMO ESTATÍSTICO AGRUPADO POR '{coluna_grupo}' ---")
    else:
        resumos = [calcular_estatisticas(df[col]) for col in colunas_analise]
        tabela = pd.DataFrame(resumos, index=colunas_analise)
        tabela.index.name = "Coluna"
        print("📊 --- RESUMO ESTATÍSTICO DAS COLUNAS NUMÉRICAS ---")

    try:
        display(tabela)
    except NameError:
        print(tabela)

    # 2. Identificação Global de Outliers
    mask_outliers_global = pd.Series(False, index=df.index)

    print("\n🚨 --- DETECÇÃO E REMOÇÃO DE OUTLIERS ---")
    for col in colunas_analise:
        q1_g = df[col].quantile(0.25)
        q3_g = df[col].quantile(0.75)
        iqr_g = q3_g - q1_g

        lim_inf_g = q1_g - 1.5 * iqr_g
        lim_sup_g = q3_g + 1.5 * iqr_g

        mask_col = (df[col] < lim_inf_g) | (df[col] > lim_sup_g)
        mask_outliers_global |= mask_col
        outliers_df = df[mask_col]

        if len(outliers_df) > 0:
            print(
                f"\nForam encontrados {len(outliers_df)} outlier(s) na coluna"
                f" '{col}':"
            )
            print(
                "• Limites válidos (IQR): Limite Inferior ="
                f" {lim_inf_g:.4f} | Limite Superior = {lim_sup_g:.4f}"
            )
            cols_exibir = [coluna_grupo, col] if coluna_grupo else [col]
            try:
                display(outliers_df[cols_exibir])
            except NameError:
                print(outliers_df[cols_exibir])
        else:
            print(f"✅ Nenhum outlier foi identificado na coluna '{col}'.")

    # 3. Exportação para arquivo
    if salvar_arquivo:
        if formato_arquivo.lower() == "excel":
            caminho_arquivo = f"{nome_arquivo}.xlsx"
            tabela.to_excel(caminho_arquivo)
            print(f"\n💾 Tabela descritiva salva em: '{caminho_arquivo}'")
        elif formato_arquivo.lower() == "csv":
            caminho_arquivo = f"{nome_arquivo}.csv"
            tabela.to_csv(caminho_arquivo, encoding="utf-8-sig")
            print(f"\n💾 Tabela descritiva salva em: '{caminho_arquivo}'")
        else:
            print(
                f"\n⚠️ Formato '{formato_arquivo}' não reconhecido. Use 'excel'"
                " ou 'csv'."
            )

    # 4. Remoção de Outliers (opcional)
    if remover_outliers:
        df_limpo = df[~mask_outliers_global].copy()
        print(
            "\n🧹 Outliers removidos! O DataFrame limpo foi retornado"
            f" ({len(df_limpo)} linhas restantes)."
        )
        return tabela, df_limpo

    return tabela